# MLflow prediction normalization

A fake already-loaded pyfunc model keeps this example offline while exercising the real Trader adapter.

In [ ]:
from datetime import UTC, datetime
from trader.predictions import FeatureBatch, FeatureColumn, FeatureRow, ModelIdentity, PredictionRequest
from trader_mlflow import MLflowPyfuncPredictor

class Frame:
    def __init__(self, rows, columns):
        self.rows = rows
        self.columns = columns

class Model:
    def predict(self, frame):
        return [[float(row[0]) * 2.0] for row in frame.rows]

now = datetime(2026, 1, 1, 12, 0, tzinfo=UTC)
batch = FeatureBatch.build(
    feature_set_id="returns-v1", feature_set_digest="sha256:features", decision_ts=now,
    schema=(FeatureColumn("return_1", "float64"),),
    rows=(
        FeatureRow(symbol="EURUSD", as_of_ts=now, availability_ts=now, values={"return_1": 0.01}),
        FeatureRow(symbol="GBPUSD", as_of_ts=now, availability_ts=now, values={"return_1": -0.02}),
    ),
)

In [ ]:
identity = ModelIdentity(
    registered_model_name="returns", model_version="1", model_version_id="returns-1",
    model_digest="sha256:model", signature_digest="sha256:signature", source_run_id="train-1",
    adapter_profile="mlflow_local_pyfunc", adapter_version="1",
)
predictor = MLflowPyfuncPredictor(
    model=Model(), dataframe_factory=Frame, identity=identity,
    output_contract=({"name": "alpha", "semantics": "expected_return", "horizon": "1bar", "units": "return"},),
)
result = predictor.predict(PredictionRequest(
    run_id="run-1", cycle_id="cycle-1", feature_batch=batch, requested_outputs=("alpha",), timeout_ms=1000,
))
assert result.status == "success"
assert [item.value for item in result.observations] == [0.02, -0.04]
assert result.feature_batch_hash == batch.input_hash